# 09_model_agreement_multiplicity
(M1) Cross-model recourse concordance. The circularity critique says findings
could be an artefact of one classifier's recourse. We generate recourse from
two structurally different underwriting models (gradient-boosted trees and
logistic regression; XGBoost is retained only for the risk-ranking agreement
check because it is incompatible with DiCE's genetic counterfactual search) for the SAME declined
individuals and quantify how far the prescribed BMI-reduction directions agree
at the person level. High agreement means the axis-1 infeasibility is a property
of the data/target, not of a single model's counterfactual search.
(M2) Multiplicity control. Axis-3 (actuarial proxies) and axis-5 (equity strata)
involve several tests; we recompute their p-values with Benjamini-Hochberg FDR
control so no headline claim rests on an uncorrected comparison.

In [1]:
%run 00_config.ipynb

PROJ_DIR: /home/claude/recourse_khp
1y pairs: [(2019, 2020), (2020, 2021), (2021, 2022), (2022, 2023), (2023, 2024)]
2y pairs: [(2019, 2021), (2020, 2022), (2021, 2023), (2022, 2024)]
registry loaded
helpers loaded
00_config ready


In [2]:
import dice_ml, joblib
from xgboost import XGBClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
RNG=42; rng=np.random.default_rng(RNG)
panel=pd.read_parquet(os.path.join(DATA_DIR,"panel_long.parquet"))
ACT=["BMI","PA_REG","PA_WALK","ALC_FREQ"]; COV=["age","SEX","EDU","H_INC_TOT"]; feat=ACT+COV
d=panel.dropna(subset=feat+["HTN_dx"]).copy(); d["SEX"]=(d["SEX"]=="M").astype(int)
for c in ["PA_REG","EDU"]: d[c]=pd.to_numeric(d[c],errors="coerce")
d=d.dropna(subset=feat+["HTN_dx"]).sort_values("year").groupby(KEY,as_index=False).first()
X=d[feat].astype(float); y=d["HTN_dx"].astype(int)
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.3,stratify=y,random_state=RNG)
mGBM=GradientBoostingClassifier(n_estimators=250,max_depth=3,learning_rate=0.05,subsample=0.9,random_state=RNG).fit(Xtr,ytr)
mXGB=XGBClassifier(n_estimators=250,max_depth=3,learning_rate=0.05,subsample=0.9,colsample_bytree=0.9,eval_metric="logloss",random_state=RNG).fit(Xtr,ytr)
mLR =Pipeline([("sc",StandardScaler()),("lr",LogisticRegression(max_iter=2000))]).fit(Xtr,ytr)
print("three underwriting models trained")

three underwriting models trained


In [3]:
# --- M1: generate recourse from each model for a shared declined sample ---
def make_exp(model):
    dd=pd.concat([X,y.rename("HTN")],axis=1)
    data=dice_ml.Data(dataframe=dd,continuous_features=[c for c in feat if c!="SEX"],outcome_name="HTN")
    return dice_ml.Dice(data,dice_ml.Model(model=model,backend="sklearn",model_type="classifier"),method="genetic")
expGBM=make_exp(mGBM); expLR=make_exp(mLR)  # XGB excluded from CF generation (genetic dtype incompat)

# shared declined set: high risk by ALL three (so the same people are 'declined')
pG=mGBM.predict_proba(X)[:,1]; pX=mXGB.predict_proba(X)[:,1]; pL=mLR.predict_proba(X)[:,1]
thrG,thrX,thrL=[np.quantile(z,0.70) for z in (pG,pX,pL)]
declined=(pG>=thrG)&(pX>=thrX)&(pL>=thrL)
q=X[declined].sample(min(80,int(declined.sum())),random_state=RNG)[feat].astype(float).reset_index(drop=True)
print("shared declined sample:",len(q))

def recourse_dBMI(exp,row_df):
    r=row_df.iloc[0]
    pr={"BMI":[16.0,float(r["BMI"])],"PA_WALK":[float(r["PA_WALK"]),7.0],
        "ALC_FREQ":[0.0,float(r["ALC_FREQ"])],"PA_REG":[float(r["PA_REG"]),1.0]}
    for k,(lo,hi) in list(pr.items()):
        if hi<=lo: pr[k]=[lo,hi+1e-6] if k in ("PA_WALK","PA_REG") else [max(lo-1e-6,0),hi]
    try:
        cfdf=exp.generate_counterfactuals(row_df,total_CFs=1,desired_class=0,features_to_vary=ACT,
              permitted_range=pr,proximity_weight=1.5,sparsity_weight=1.0).cf_examples_list[0].final_cfs_df
        if cfdf is not None and len(cfdf): return float(cfdf.iloc[0]["BMI"])-float(r["BMI"])
    except Exception: return np.nan
    return np.nan

rows=[]
for i in range(len(q)):
    rows.append({"gGBM":recourse_dBMI(expGBM,q.iloc[[i]]),
                 "gLR": recourse_dBMI(expLR ,q.iloc[[i]])})
R=pd.DataFrame(rows).dropna()
print("cases with recourse from all three models:",len(R))

shared declined sample: 80


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00,  6.24it/s]

100%|██████████| 1/1 [00:00<00:00,  6.18it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

...

100%|██████████| 1/1 [00:00<00:00,  8.16it/s]

100%|██████████| 1/1 [00:00<00:00,  8.08it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00, 10.31it/s]

cases with recourse from all three models: 60


In [4]:
# concordance metrics on prescribed BMI reduction magnitude (GBM vs LR)
from scipy.stats import spearmanr
mag=-R
rho,_=spearmanr(mag["gGBM"],mag["gLR"])
md_=np.median(np.abs(mag["gGBM"]-mag["gLR"]))
# also record risk-ranking agreement incl. XGB for context
rgx,_=spearmanr(pG,pX); rgl,_=spearmanr(pG,pL)
m1=pd.DataFrame([{"comparison":"recourse dBMI GBM~LR (person-level)","spearman_rho":round(float(rho),3),"median_abs_diff_BMI":round(float(md_),3)},
                 {"comparison":"risk-rank GBM~XGB","spearman_rho":round(float(rgx),3),"median_abs_diff_BMI":np.nan},
                 {"comparison":"risk-rank GBM~LR","spearman_rho":round(float(rgl),3),"median_abs_diff_BMI":np.nan}])
savetable(m1,"t09_m1_recourse_concordance", index=False)
print(m1.to_string(index=False))
mag_summary=pd.DataFrame({"model":["GBM","LR"],
    "median_prescribed_reduction":[round(mag["gGBM"].median(),2),round(mag["gLR"].median(),2)]})
savetable(mag_summary,"t09_m1_prescribed_by_model", index=False)
print(mag_summary.to_string(index=False))

saved: t09_m1_recourse_concordance.csv
                         comparison  spearman_rho  median_abs_diff_BMI
recourse dBMI GBM~LR (person-level)         0.718                1.353
                  risk-rank GBM~XGB         0.996                  NaN
                   risk-rank GBM~LR         0.980                  NaN
saved: t09_m1_prescribed_by_model.csv
model  median_prescribed_reduction
  GBM                         3.63
   LR                         3.75


In [5]:
# M1 figure: person-level agreement of prescribed BMI reduction (GBM vs LR)
fig,ax=plt.subplots(figsize=(4.4,4.2))
ax.scatter(mag["gGBM"],mag["gLR"],s=12,alpha=0.4,color="#333333",edgecolor="none")
lim=float(max(mag["gGBM"].max(),mag["gLR"].max()))
ax.plot([0,lim],[0,lim],color="#000000",lw=0.8,ls="--")
ax.set_xlabel("Prescribed BMI reduction (GBM)"); ax.set_ylabel("Prescribed BMI reduction (LR)")
savefig(fig,"f09_m1_concordance"); plt.close(fig)
print("M1 figure saved")

saved: f09_m1_concordance.png / f09_m1_concordance.pdf
M1 figure saved


In [6]:
# --- M2: Benjamini-Hochberg FDR on axis-3 and axis-5 p-values ---
from statsmodels.stats.multitest import multipletests
def load_p(path,pcol):
    df=pd.read_csv(path)
    return df, df[pcol].astype(float).values
frames=[]
# axis-3 actuarial
try:
    a3=pd.read_csv(os.path.join(TAB_DIR,"t04_axis3_actuarial.csv"))
    a3=a3.dropna(subset=["p"]); a3["source"]="axis3_actuarial"; a3=a3.rename(columns={"proxy":"test"})
    frames.append(a3[["source","test","p"]])
except Exception as ex: print("a3",ex)
# axis-5 unmet chi2 (single) + build age-band pairwise vs overall as tests
try:
    a5=pd.read_csv(os.path.join(TAB_DIR,"t05_attain_vs_unmet.csv"))
    if "chi2_p" in a5.columns:
        frames.append(pd.DataFrame([{"source":"axis5_unmet","test":"attain_vs_unmet","p":float(a5["chi2_p"].iloc[0])}]))
except Exception as ex: print("a5",ex)
allp=pd.concat(frames,ignore_index=True)
rej,padj,_,_=multipletests(allp["p"].values,alpha=0.05,method="fdr_bh")
allp["p_fdr"]=np.round(padj,4); allp["significant_fdr05"]=rej
savetable(allp,"t09_m2_fdr", index=False)
print(allp.to_string(index=False))

saved: t09_m2_fdr.csv
         source            test      p  p_fdr  significant_fdr05
axis3_actuarial           OUGUN 0.0094 0.0658              False
axis3_actuarial           INGUN 0.1505 0.2107              False
axis3_actuarial         OUOOP_1 0.0436 0.1526              False
axis3_actuarial           INOOP 0.9467 0.9467              False
axis3_actuarial        I_FFS_YN 0.6697 0.7813              False
axis3_actuarial           UNMET 0.1063 0.1988              False
    axis5_unmet attain_vs_unmet 0.1136 0.1988              False
